In [1]:
pip install gradio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\SATISFY\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODELS_URL = "https://openrouter.ai/api/v1/models"


C:\Users\SATISFY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
test_headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}"
}

test = requests.get(
    "https://openrouter.ai/api/v1/auth/key",
    headers=test_headers
)

print(test.status_code)
print(test.text)

200
{"data":{"label":"sk-or-v1-d93...062","is_management_key":false,"is_provisioning_key":false,"limit":null,"limit_reset":null,"limit_remaining":null,"include_byok_in_limit":false,"usage":0,"usage_daily":0,"usage_weekly":0,"usage_monthly":0,"byok_usage":0,"byok_usage_daily":0,"byok_usage_weekly":0,"byok_usage_monthly":0,"is_free_tier":false,"expires_at":"2026-09-18T11:50:26.821Z","creator_user_id":"user_3A0Aw6gLSFsOE3YCTxYTwqMGRJY","rate_limit":{"requests":-1,"interval":"10s","note":"This field is deprecated and safe to ignore."}}}


In [4]:
print("Key loaded:", OPENROUTER_API_KEY is not None)
print("Key format:", OPENROUTER_API_KEY.startswith("sk-or-"))
print("Key length:", len(OPENROUTER_API_KEY))

Key loaded: True
Key format: True
Key length: 73


In [5]:
test_headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json"
}

test_data = {
    "model": "openai/gpt-oss-20b:free",
    "messages": [
        {
            "role": "user",
            "content": "Say hello"
        }
    ]
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers=test_headers,
    json=test_data
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 404
Response: {"error":{"message":"This model is unavailable for free. The paid version is available now - use this slug instead: openai/gpt-oss-20b","code":404},"user_id":"user_3A0Aw6gLSFsOE3YCTxYTwqMGRJY"}


In [6]:
print("Key loaded:", OPENROUTER_API_KEY is not None)
print("Key starts correctly:", OPENROUTER_API_KEY.startswith("sk-or-") if OPENROUTER_API_KEY else False)

test_headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json"
}

print("Authorization header created:", "Authorization" in test_headers)
print("Header starts with Bearer:", test_headers["Authorization"].startswith("Bearer sk-or-"))

Key loaded: True
Key starts correctly: True
Authorization header created: True
Header starts with Bearer: True


In [7]:
from dotenv import find_dotenv

print("Current folder:", os.getcwd())
print(".env found at:", find_dotenv())

Current folder: c:\Users\SATISFY\OneDrive\Desktop\jekacode-ai-engineering\week03-llm-apis
.env found at: c:\Users\SATISFY\OneDrive\Desktop\jekacode-ai-engineering\.env


In [8]:
from dotenv import dotenv_values

env_values = dotenv_values(find_dotenv())

print("Variables found in .env:")
print(list(env_values.keys()))

Variables found in .env:
['GEMINI_API_KEY', 'GEMINI_MODEL', 'GROQ_API_KEY', 'GROQ_MODEL', 'DEEPSEEK_API_KEY', 'HF_TOKEN', 'HF_MODEL', 'OLLAMA_MODEL', 'OLLAMA_URL', 'OPENROUTER_API_KEY']


In [9]:
def list_free_models():
    try:
        r = requests.get(MODELS_URL, timeout=15)
        r.raise_for_status()
        all_models = r.json()["data"]
        free_ids = [m["id"] for m in all_models if m["id"].endswith(":free")]
        free_ids.sort()
    except Exception:
        free_ids = []
    # openrouter/free is a built-in router that always works, so it's always first
    return ["openrouter/free"] + free_ids


In [10]:
def ask_openrouter(messages, model):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "AI Chatbot",
    }
    payload = {"model": model, "messages": messages}
    r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

In [11]:
latest_reponses = {}

In [12]:
def chat_fn(message, history, models):
    global latest_reponses

    if not models:
        return "Please select at least one model."

    messages = []

    for turn in history:
        messages.append({
            "role": turn["role"],
            "content": turn["content"]
        })

    messages.append({
        "role": "user",
        "content": message
    })

    responses = {}

    for model in models:
        try:
            response = ask_openrouter(messages, model)
            responses[model] = response

        except requests.exceptions.HTTPError as e:
            responses[model] = f"Error: {e}"

        except Exception as e:
            responses[model] = f"Something went wrong: {e}"
            latest_reponses = responses

    return "\n\n".join(
        f"### {model}\n{response}"
        for model, response in responses.items()
    )

In [13]:
def cross_analyze(responses):
    analysis = "\n\n".join(
        f"MODEL: {model}\nRESPONSE:\n{response}"
        for model, response in responses.items()
    )

    prompt = f"""
Compare the following AI model responses.

{analysis}

Give a concise cross-analysis covering:
1. Accuracy
2. Clarity
3. Completeness
4. Strengths
5. Weaknesses

Finally, state which response is strongest overall and briefly explain why.
"""

    return ask_openrouter(
        [{"role": "user", "content": prompt}],
        "openrouter/free"
    )

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [15]:
print(free_models := list_free_models())

['openrouter/free', 'cohere/north-mini-code:free', 'dots-studio/dots-3-note-preview:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'inclusionai/ling-3.0-flash-fin:free', 'inclusionai/ling-3.0-flash-sante:free', 'inclusionai/ling-3.0-flash-vl:free', 'liquid/lfm-2.5-2.6b:free', 'nex-agi/nex-n2.5-mini:free', 'nex-agi/nex-n2.5-pro:free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'nvidia/nemotron-3-super-120b-a12b:free', 'nvidia/nemotron-3-ultra-550b-a55b:free', 'nvidia/nemotron-3.5-content-safety:free', 'nvidia/nemotron-3.5-lightning:free', 'poolside/laguna-s-2.1:free', 'poolside/laguna-xs-2.1:free', 'thinkingmachines/inkling-small:free', 'thinkingmachines/inkling:free']


In [16]:
free_models = list_free_models()
default_model = free_models[0] if free_models else "openrouter/free"

with gr.Blocks() as demo:

    model_picker = gr.Dropdown(
        choices=free_models,
        value=[default_model],
        multiselect=True,
        label="Select Models (free tier)",
    )

    chat = gr.ChatInterface(
        fn=chat_fn,
        additional_inputs=[model_picker],
        title="AI Chatbot — free models via OpenRouter",
        description="Pick free models, then chat.",
    )

    cross_analysis_btn = gr.Button("🔍 Run Cross-Analysis")
    clear_btn = gr.Button("🗑️ Clear Chat")

    analysis_output = gr.Markdown()

    cross_analysis_btn.click(
        fn=lambda: cross_analyze(latest_responses),
        inputs=[],
        outputs=analysis_output
    )
    clear_btn.click(
    fn=None,
    inputs=[],
    outputs=[],
    js="() => { document.querySelectorAll('button').forEach(b => { if (b.innerText.includes('Clear')) b.click(); }); }"
)
    

    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
